# Notebook 3 — 코사인 유사도 + Gradio UI

`data/stats.csv`를 기반으로 선수 간 코사인 유사도를 계산하고,
4개 리그 (EPL / La Liga / Bundesliga / Serie A) 에서 유사 선수를 찾는 웹 UI를 실행합니다.

**입력:** `data/stats.csv`

## 패키지 설치

In [ ]:
!pip install gradio fuzzywuzzy python-Levenshtein scikit-learn -q

## 라이브러리 임포트 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from fuzzywuzzy import fuzz
import gradio as gr
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(os.path.join('data', 'stats.csv'))
print(f'로드 완료: {df.shape}')
print(f'컬럼: {df.columns.tolist()[:10]} ...')
df.head(3)

## 코사인 유사도 행렬 계산

In [ ]:
# 메타 컬럼 제외 — 숫자 피처만 사용
META_COLS = {'ID', 'League', 'Season', 'Team', 'Player', 'Pos', 'Nation', 'Age', 'Born'}
feature_cols = [c for c in df.columns if c not in META_COLS and df[c].dtype.kind in 'fiu']
print(f'피처 컬럼 수: {len(feature_cols)}')

X = df[feature_cols].values
sim_matrix = cosine_similarity(X)

# ID를 인덱스/컬럼으로 하는 DataFrame
id_list = df['ID'].tolist()
cosim_df = pd.DataFrame(sim_matrix, index=id_list, columns=id_list)
print(f'유사도 행렬: {cosim_df.shape}')

## 선수 이름 검색 헬퍼

In [ ]:
# 리그별 선수 이름 목록 (중복 제거)
all_names = df['Player'].dropna().unique().tolist()
df_names = pd.DataFrame(all_names, columns=['Player'])

def fuzzy_match_name(query: str) -> str:
    """오타를 허용하여 가장 비슷한 선수 이름을 반환"""
    scores = df_names['Player'].apply(lambda n: fuzz.ratio(query.lower(), n.lower()))
    best_idx = scores.idxmax()
    return df_names.loc[best_idx, 'Player']

# 테스트
print(fuzzy_match_name('son heung min'))   # → Son Heung-min
print(fuzzy_match_name('messi'))           # → Lionel Messi (또는 유사 이름)
print(fuzzy_match_name('lewandowski'))     # → Robert Lewandowski

## 유사 선수 검색 함수

In [ ]:
# ID 형식: {LeagueShort}_{Season}_{Player}
# 예: EPL_2021-22_Son Heung-min  /  LaLiga_2022-23_Lionel Messi
LEAGUE_DISPLAY = {
    'EPL':        'ENG-Premier League',
    'La Liga':    'ESP-La Liga',
    'Bundesliga': 'GER-Bundesliga',
    'Serie A':    'ITA-Serie A',
}
LEAGUE_SHORT_REV = {
    'ENG-Premier League': 'EPL',
    'ESP-La Liga':        'LaLiga',
    'GER-Bundesliga':     'Bundesliga',
    'ITA-Serie A':        'SerieA',
}

def find_similar_players(league: str, season: str, player_name: str) -> pd.DataFrame:
    """
    league    : 'EPL' / 'La Liga' / 'Bundesliga' / 'Serie A'
    season    : '2017-18' ~ '2023-24'
    player_name: 선수 이름 (오타 허용)
    """
    matched_name = fuzzy_match_name(player_name)

    # ID에 사용할 리그 약칭
    league_full = LEAGUE_DISPLAY.get(league, league)
    league_code = LEAGUE_SHORT_REV.get(league_full, league)

    player_id = f'{league_code}_{season}_{matched_name}'

    if player_id not in cosim_df.columns:
        available = [c for c in cosim_df.columns if matched_name in c]
        if not available:
            return pd.DataFrame({'Error': [f'"{matched_name}" 를 찾을 수 없습니다.']})
        player_id = available[0]  # 같은 선수의 다른 시즌으로 fallback

    top10 = cosim_df[player_id].nlargest(n=10)
    result = top10.reset_index()
    result.columns = ['ID', 'Similarity']

    # ID에서 메타 정보 파싱하여 컬럼 추가
    id_meta = df.set_index('ID')[['Player', 'League', 'Season', 'Team', 'Pos']]
    result = result.join(id_meta, on='ID')
    result['Similarity'] = result['Similarity'].round(4)

    return result[['Player', 'League', 'Season', 'Team', 'Pos', 'Similarity']]


# 테스트
find_similar_players('EPL', '2021-22', 'son heung min')

## Gradio UI 실행

In [ ]:
LEAGUES = ['EPL', 'La Liga', 'Bundesliga', 'Serie A']
SEASONS = ['2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']

demo = gr.Interface(
    fn=find_similar_players,
    inputs=[
        gr.Radio(LEAGUES, label='League', value='EPL'),
        gr.Radio(SEASONS, label='Season', value='2021-22'),
        gr.Textbox(label='Player Name', placeholder='예: Son Heung-min, Messi, Lewandowski ...'),
    ],
    outputs=gr.Dataframe(label='Top 9 Similar Players'),
    title='Soccer Similar Players',
    description='4개 유럽 리그(2017-18 ~ 2023-24) 선수 통계 기반 코사인 유사도 검색',
    live=False,
)

demo.launch(share=True)